# YouTube Trending: Category Classification & View-Count Regression with PySpark
**BAN5600 – Advanced Big Data Computing & Programming** | Mark Hapaguti

Follows CRISP-DM. **Research Q1:** predict `category` from title, tags, channel and publication-timing metadata (no `category_id` as a feature). **Research Q2:** predict `log(1+views)` from metadata and timing only (no likes / dislikes / comments).

**Sections:** 0 Setup · 1 Load data · 2 Clean & integrate · 3 Descriptive stats & EDA · 4 Feature engineering · 5 Classification · 6 Regression · 7 Save outputs

**Data files needed** (from the Kaggle dataset `datasnaek/youtube-new`): `CAvideos.csv`, `USvideos.csv`, … and `CA_category_id.json`, `US_category_id.json`, … The notebook uses whichever regions it finds, so you can test with one region and then run all ten.

## 0. Setup

In [ ]:
!pip -q install pyspark seaborn scikit-learn

In [ ]:
import os, time, glob, shutil, warnings
from functools import reduce
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (RegexTokenizer, StopWordsRemover, CountVectorizer, IDF,
                                StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler)
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier, OneVsRest
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# ---------------- CONFIG (edit these) ----------------
DATA_DIR    = "/content/data"      # where the CSV/JSON files live
FIG_DIR     = "/content/figures"
OUT_DIR     = "/content/outputs"
ALL_REGIONS = ["US","GB","DE","CA","FR","RU","MX","KR","JP","IN"]
SEED        = 42
TRAIN_PCT   = 80                   # % of *videos* in the training set
MIN_CLASS   = 300                  # drop categories with fewer records than this (classification)
VOCAB_SIZE  = 3000                 # text features (TF-IDF vocabulary)
RF_TREES, RF_DEPTH   = 60, 12
RUN_GBT_CLASSIFIER   = False       # One-vs-Rest GBT is the slowest step; set False to skip
RUN_TUNING           = False       # optional CrossValidator on the regression Random Forest
# ------------------------------------------------------
for d in (DATA_DIR, FIG_DIR, OUT_DIR): os.makedirs(d, exist_ok=True)

spark = (SparkSession.builder.master("local[*]").appName("YouTubeTrending_BAN5600")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism)

def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG_DIR}/{name}.png", dpi=200, bbox_inches="tight"); plt.show()

## 1. Load data
Upload the CSV and JSON files (or a single Kaggle `archive.zip`). If they are already in `DATA_DIR` (e.g., copied from Google Drive), this cell does nothing.

In [ ]:
def have_files():
    return glob.glob(f"{DATA_DIR}/*videos.csv")

if not have_files():
    try:
        from google.colab import files
        print("Select the *videos.csv and *_category_id.json files (or archive.zip)...")
        up = files.upload()
        for fn in up:
            if fn.endswith(".zip"):
                shutil.unpack_archive(fn, DATA_DIR)
            else:
                shutil.move(fn, os.path.join(DATA_DIR, fn))
    except ImportError:
        raise SystemExit(f"Not on Colab: put the files in {DATA_DIR}")

# Option: Google Drive instead of upload
# from google.colab import drive; drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/youtube-new'

# some Kaggle zips nest files in a folder - flatten
for p in glob.glob(f"{DATA_DIR}/**/*videos.csv", recursive=True) + glob.glob(f"{DATA_DIR}/**/*_category_id.json", recursive=True):
    if os.path.dirname(p) != DATA_DIR: shutil.move(p, os.path.join(DATA_DIR, os.path.basename(p)))

REGIONS = [r for r in ALL_REGIONS if os.path.exists(f"{DATA_DIR}/{r}videos.csv")]
print("Regions found:", REGIONS)
assert REGIONS, "No *videos.csv files found"

In [ ]:
def load_region(r):
    df = (spark.read.option("header", True).option("multiLine", True)
          .option("quote", '"').option("escape", '"').option("mode", "PERMISSIVE")
          .csv(f"{DATA_DIR}/{r}videos.csv"))
    return df.withColumn("country", F.lit(r))

raw = reduce(DataFrame.unionByName, [load_region(r) for r in REGIONS]).cache()
print("Raw records:", raw.count()); raw.printSchema()
raw.groupBy("country").count().orderBy("country").show()

## 2. Clean & integrate (Data Preparation)
Steps: cast types, parse `trending_date` (format `yy.dd.mm`) and `publish_time`, drop malformed rows left by multi-line fields, join category labels from the region JSON files, and keep **one row per (video, country)**. A video appears on many trending days with growing view counts, so we keep the row with the highest views.

In [ ]:
# Null counts on the raw string columns (for the missing-data chart)
key_cols = ["video_id","trending_date","title","channel_title","category_id","publish_time","tags","views","likes","dislikes","comment_count","description"]
nulls_raw = raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in key_cols]).toPandas().T[0]

df = raw
for c in ["category_id","views","likes","dislikes","comment_count"]:
    df = df.withColumn(c, F.col(c).cast("long"))
for c in ["comments_disabled","ratings_disabled","video_error_or_removed"]:
    df = df.withColumn(c, (F.col(c) == "True").cast("int"))
df = (df.withColumn("trending_date", F.to_date("trending_date", "yy.dd.MM"))
        .withColumn("publish_time",  F.col("publish_time").cast("timestamp")))

# drop malformed rows (bad ids like '#NAME?', unparseable numbers/dates)
df = df.filter(F.col("video_id").rlike(r"^[\w-]{11}$") & F.col("category_id").isNotNull() &
               F.col("views").isNotNull() & (F.col("views") > 0) &
               F.col("publish_time").isNotNull() & F.col("trending_date").isNotNull())
n_clean = df.count(); print("After cleaning:", n_clean, "| dropped:", raw.count() - n_clean)

In [ ]:
# ---- category labels from the region JSON files (fallback: standard YouTube mapping) ----
cat_parts = []
for r in REGIONS:
    p = f"{DATA_DIR}/{r}_category_id.json"
    if os.path.exists(p):
        j = spark.read.option("multiLine", True).json(p)
        cat_parts.append(j.select(F.explode("items").alias("i"))
                          .select(F.col("i.id").cast("int").alias("category_id"), F.col("i.snippet.title").alias("category")))
if cat_parts:
    cats = reduce(DataFrame.unionByName, cat_parts).dropDuplicates(["category_id"])
    print("Category labels loaded from JSON files")
else:
    fallback = {1:"Film & Animation",2:"Autos & Vehicles",10:"Music",15:"Pets & Animals",17:"Sports",18:"Short Movies",
        19:"Travel & Events",20:"Gaming",21:"Videoblogging",22:"People & Blogs",23:"Comedy",24:"Entertainment",
        25:"News & Politics",26:"Howto & Style",27:"Education",28:"Science & Technology",29:"Nonprofits & Activism",
        30:"Movies",43:"Shows",44:"Trailers"}
    cats = spark.createDataFrame(list(fallback.items()), ["category_id","category"])
    print("JSON files not found - using standard YouTube category mapping")

df = df.join(F.broadcast(cats), "category_id", "left").withColumn("category", F.coalesce("category", F.concat(F.lit("id_"), F.col("category_id").cast("string"))))

# keep one row per (video, country): the latest/highest view count
w = Window.partitionBy("video_id", "country").orderBy(F.col("views").desc(), F.col("trending_date").desc())
all_records = df.cache()                                   # every trending record (for EDA)
videos = (df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")).cache()
print("Trending records:", all_records.count(), "| unique (video, country):", videos.count())

## 3. Descriptive statistics & data visualizations (Data Understanding)

In [ ]:
# 3.1 Summary statistics (computed in Spark)
num = ["views","likes","dislikes","comment_count"]
summ = all_records.select(num).summary("count","mean","stddev","min","25%","50%","75%","max").toPandas().set_index("summary").T
summ = summ.astype(float).round(1); display(summ)
summ.to_csv(f"{OUT_DIR}/descriptive_statistics.csv")

print("Skewness of views (raw vs log):")
all_records.select(F.skewness("views").alias("raw"), F.skewness(F.log10("views")).alias("log10")).show()

In [ ]:
# 3.2 Records by country  |  3.3 Missing values (raw)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
cc = all_records.groupBy("country").count().toPandas().sort_values("count", ascending=False)
sns.barplot(data=cc, x="country", y="count", color="#2b6cb0", ax=ax[0]); ax[0].set_title("Trending records by country"); ax[0].set_ylabel("Records")
miss = (nulls_raw / raw.count() * 100).sort_values(ascending=False); miss = miss[miss > 0]
if len(miss): miss[::-1].plot.barh(ax=ax[1], color="#c05621")
ax[1].set_title("Missing values in raw data (%)"); ax[1].set_xlabel("% of records")
savefig("01_country_missing")

In [ ]:
# 3.4 Distribution of views (log scale) - computed in Spark, then plotted
hist = (all_records.select((F.floor(F.log10("views") * 10) / 10).alias("bin")).groupBy("bin").count().orderBy("bin").toPandas())
fig, ax = plt.subplots(figsize=(7, 3.6)); ax.bar(hist["bin"], hist["count"], width=0.095, color="#2b6cb0")
ax.set_xlabel("log10(views)"); ax.set_ylabel("Records"); ax.set_title("Distribution of view counts (log scale)")
savefig("02_views_distribution")

In [ ]:
# 3.5 Category distribution overall and by country
cat_cnt = videos.groupBy("category").count().toPandas().sort_values("count", ascending=False)
cat_cc  = videos.groupBy("category", "country").count().toPandas()
fig, ax = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1, 1.2]})
sns.barplot(data=cat_cnt.head(15), y="category", x="count", color="#2b6cb0", ax=ax[0]); ax[0].set_title("Videos by category (top 15)")
top = cat_cnt.head(12)["category"]
pv = cat_cc[cat_cc.category.isin(top)].pivot(index="category", columns="country", values="count").fillna(0)
pv = pv.div(pv.sum(axis=0), axis=1).loc[top] * 100
sns.heatmap(pv, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% of country's videos"}, ax=ax[1]); ax[1].set_title("Category mix by country (%)"); ax[1].set_ylabel("")
savefig("03_categories")

In [ ]:
# 3.6 Engagement by category and country
med_cat = (videos.groupBy("category").agg(F.percentile_approx("views", 0.5).alias("median_views"), F.count("*").alias("n"))
           .filter("n >= 100").toPandas().sort_values("median_views"))
med_cty = videos.groupBy("country").agg(F.percentile_approx("views", 0.5).alias("median_views")).toPandas().sort_values("median_views")
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.4, 1]})
ax[0].barh(med_cat["category"], med_cat["median_views"] / 1e3, color="#c05621"); ax[0].set_xlabel("Median views (thousands)"); ax[0].set_title("Median views by category (n ≥ 100)")
ax[1].barh(med_cty["country"], med_cty["median_views"] / 1e3, color="#2f855a"); ax[1].set_xlabel("Median views (thousands)"); ax[1].set_title("Median views by country")
savefig("04_engagement_by_category_country")

# distribution of log-views per category (sampled for plotting)
samp = videos.select("category", F.log10("views").alias("log_views")).sample(0.25, seed=SEED).limit(60000).toPandas()
order = samp.groupby("category").log_views.median().sort_values(ascending=False).index
keep = [c for c in order if (samp.category == c).sum() > 50][:14]
fig, ax = plt.subplots(figsize=(11, 4.5)); sns.boxplot(data=samp[samp.category.isin(keep)], x="category", y="log_views", order=keep, color="#9ecae1", fliersize=1, ax=ax)
ax.set_title("log10(views) by category"); ax.tick_params(axis="x", rotation=60); ax.set_xlabel("")
savefig("05_logviews_boxplot")

In [ ]:
# 3.7 Publication timing and time-to-trend
t = videos.withColumn("pub_hour", F.hour("publish_time")).withColumn("pub_dow", F.dayofweek("publish_time")) \
          .withColumn("days_to_trend", F.datediff("trending_date", F.to_date("publish_time")))
hour = t.groupBy("pub_hour").agg(F.count("*").alias("n"), F.avg(F.log10("views")).alias("avg_log_views")).orderBy("pub_hour").toPandas()
dow  = t.groupBy("pub_dow").agg(F.count("*").alias("n"), F.avg(F.log10("views")).alias("avg_log_views")).orderBy("pub_dow").toPandas()
dow["day"] = dow.pub_dow.map({1:"Sun",2:"Mon",3:"Tue",4:"Wed",5:"Thu",6:"Fri",7:"Sat"})
ttt = t.filter("days_to_trend between 0 and 30").groupBy("days_to_trend").count().orderBy("days_to_trend").toPandas()
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].bar(hour.pub_hour, hour.n, color="#2f855a"); ax[0].set_title("Videos by publish hour (UTC)"); ax[0].set_xlabel("Hour")
ax[1].bar(dow.day, dow.n, color="#2b6cb0"); ax[1].set_title("Videos by publish weekday")
ax[2].bar(ttt.days_to_trend, ttt["count"], color="#805ad5"); ax[2].set_title("Days from publication to first trending record*"); ax[2].set_xlabel("Days")
savefig("06_timing")
print("*Descriptive only. days_to_trend is NOT used as a model feature (it is only known after the video trends).")

In [ ]:
# 3.8 Correlations, text-metadata relationships, top channels
sm = (videos.select("views","likes","dislikes","comment_count", F.length("title").alias("title_len"),
                    F.when(F.col("tags") == "[none]", 0).otherwise(F.size(F.split(F.coalesce("tags", F.lit("")), r"\|"))).alias("tag_count"))
      .sample(0.5, seed=SEED).limit(80000).toPandas())
lg = sm.copy()
for c in ["views","likes","dislikes","comment_count"]: lg[c] = np.log1p(lg[c])
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
sns.heatmap(lg.corr(), annot=True, fmt=".2f", cmap="Blues", ax=ax[0]); ax[0].set_title("Correlation (log1p engagement)")
ax[1].hexbin(sm.tag_count.clip(upper=60), lg.views, gridsize=35, cmap="Blues", mincnt=1); ax[1].set_xlabel("Tag count"); ax[1].set_ylabel("log(1+views)"); ax[1].set_title("Tag count vs views")
ax[2].hexbin(sm.title_len, lg.views, gridsize=35, cmap="Blues", mincnt=1); ax[2].set_xlabel("Title length"); ax[2].set_title("Title length vs views")
savefig("07_correlations")

top_ch = all_records.groupBy("channel_title").agg(F.count("*").alias("records"), F.countDistinct("video_id").alias("videos")).orderBy(F.desc("records")).limit(15).toPandas()
fig, ax = plt.subplots(figsize=(8, 4.5)); sns.barplot(data=top_ch, y="channel_title", x="records", color="#2b6cb0", ax=ax); ax.set_title("Top 15 channels by trending records"); ax.set_ylabel("")
savefig("08_top_channels")

## 4. Feature engineering & leakage-safe split (Data Preparation)
* **Metadata features:** title length / words / uppercase ratio / `!` `?` digits, tag count, description length & link flag, publish hour / weekday / month, comments- & ratings-disabled flags, channel frequency (from training data only), country.
* **Text features:** TF-IDF over title + tags (tokenizer → stop words → CountVectorizer → IDF; Unicode-aware so RU/KR/JP text survives).
* **Excluded to prevent leakage:** `views`, `likes`, `dislikes`, `comment_count` (regression predictors), `category_id` (classification predictor), `trending_date` and days-to-trend (only known after the video trends).
* **Split:** by hashing `video_id`, so a video (in any country) is *entirely* in train or in test.

In [ ]:
tags_clean = F.regexp_replace(F.regexp_replace(F.coalesce("tags", F.lit("")), '"', ""), r"\|", " ")
feat = (videos
    .withColumn("title_len",   F.length("title").cast("double"))
    .withColumn("title_words", F.size(F.split(F.trim("title"), r"\s+")).cast("double"))
    .withColumn("title_upper_ratio", F.length(F.regexp_replace("title", "[^A-Z]", "")) / F.greatest(F.length("title"), F.lit(1)))
    .withColumn("title_has_excl",  F.col("title").contains("!").cast("double"))
    .withColumn("title_has_q",     F.col("title").contains("?").cast("double"))
    .withColumn("title_has_digit", F.col("title").rlike(r"\d").cast("double"))
    .withColumn("tag_count", F.when(F.coalesce("tags", F.lit("[none]")) == "[none]", 0).otherwise(F.size(F.split("tags", r"\|"))).cast("double"))
    .withColumn("desc_len",  F.length(F.coalesce("description", F.lit(""))).cast("double"))
    .withColumn("desc_has_link", F.coalesce("description", F.lit("")).contains("http").cast("double"))
    .withColumn("pub_hour",  F.hour("publish_time").cast("double"))
    .withColumn("pub_dow",   F.dayofweek("publish_time").cast("double"))
    .withColumn("pub_month", F.month("publish_time").cast("double"))
    .withColumn("comments_disabled", F.col("comments_disabled").cast("double"))
    .withColumn("ratings_disabled",  F.col("ratings_disabled").cast("double"))
    .withColumn("text", F.lower(F.concat_ws(" ", "title", tags_clean)))
    .withColumn("log_views", F.log1p("views"))
    .withColumn("bucket", F.abs(F.xxhash64("video_id")) % 100))

train_raw = feat.filter(F.col("bucket") <  TRAIN_PCT)
test_raw  = feat.filter(F.col("bucket") >= TRAIN_PCT)

# channel frequency from TRAIN only (unseen channels -> 0)
ch = train_raw.groupBy("channel_title", "country").agg(F.log1p(F.count("*")).alias("channel_video_count"))
add_ch = lambda d: d.join(ch, ["channel_title", "country"], "left").fillna({"channel_video_count": 0.0})
train_raw, test_raw = add_ch(train_raw), add_ch(test_raw)

NUM_COLS = ["title_len","title_words","title_upper_ratio","title_has_excl","title_has_q","title_has_digit",
            "tag_count","desc_len","desc_has_link","pub_hour","pub_dow","pub_month",
            "comments_disabled","ratings_disabled","channel_video_count"]

feat_pipe = Pipeline(stages=[
    RegexTokenizer(inputCol="text", outputCol="tokens", pattern=r"[^\p{L}\p{N}]+", gaps=True, minTokenLength=2),
    StopWordsRemover(inputCol="tokens", outputCol="tokens_clean"),
    CountVectorizer(inputCol="tokens_clean", outputCol="tf", vocabSize=VOCAB_SIZE, minDF=10),
    IDF(inputCol="tf", outputCol="tfidf"),
    VectorAssembler(inputCols=NUM_COLS, outputCol="num_raw"),
    StandardScaler(inputCol="num_raw", outputCol="num_scaled", withMean=True, withStd=True),
    StringIndexer(inputCol="country", outputCol="country_idx", handleInvalid="keep"),
    OneHotEncoder(inputCols=["country_idx"], outputCols=["country_vec"], dropLast=False),
    VectorAssembler(inputCols=["num_scaled", "country_vec", "tfidf"], outputCol="features")])
feat_model = feat_pipe.fit(train_raw)                      # fitted on TRAIN only
train = feat_model.transform(train_raw).cache()
test  = feat_model.transform(test_raw).cache()

# human-readable feature names (for importance plots)
st = feat_model.stages
country_labels = list(st[6].labelsArray[0]) + ["unseen"]
FEATURE_NAMES = NUM_COLS + [f"country={c}" for c in country_labels] + [f"word:{w}" for w in st[2].vocabulary]
n_feat = train.select("features").first()[0].size
FEATURE_NAMES = (FEATURE_NAMES + [f"f{i}" for i in range(len(FEATURE_NAMES), n_feat)])[:n_feat]
print(f"Train videos: {train.count():,} | Test videos: {test.count():,} | Features: {n_feat:,}")

## 5. Research Question 1 – Category classification (Modeling & Evaluation)
Models: **Multinomial Logistic Regression** (baseline), **Random Forest**, **One-vs-Rest Gradient-Boosted Trees** (MLlib's GBT classifier is binary-only). Class weights counter the imbalance (Entertainment dominates). Because of the imbalance, **macro-F1** is reported alongside accuracy and weighted-F1.

In [ ]:
# keep categories with enough records, index the label, add balanced class weights
big = [r["category"] for r in videos.groupBy("category").count().filter(F.col("count") >= MIN_CLASS).collect()]
print(f"{len(big)} categories kept:", sorted(big))
tr_c = train.filter(F.col("category").isin(big)); te_c = test.filter(F.col("category").isin(big))
lab_model = StringIndexer(inputCol="category", outputCol="label", handleInvalid="skip").fit(tr_c)
CLASS_NAMES = lab_model.labels
tr_c, te_c = lab_model.transform(tr_c), lab_model.transform(te_c)
cnt = tr_c.groupBy("label").count(); total, k = tr_c.count(), cnt.count()
tr_c = tr_c.join(cnt.select("label", (F.lit(total) / (F.lit(k) * F.col("count"))).alias("weight")), "label").cache()
te_c = te_c.cache(); print("Train:", tr_c.count(), "| Test:", te_c.count())

In [ ]:
clf_results, clf_models, clf_preds = [], {}, {}

def eval_clf(name, model, secs):
    pred = model.transform(te_c); clf_preds[name] = pred
    res = {"model": name, "train_sec": round(secs, 1)}
    for m, key in [("accuracy","accuracy"), ("f1","weighted_f1"), ("weightedPrecision","w_precision"), ("weightedRecall","w_recall")]:
        res[key] = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=m).evaluate(pred)
    pdf = pred.select("label", "prediction").toPandas()
    res["macro_f1"] = f1_score(pdf.label, pdf.prediction, average="macro")
    clf_results.append(res); clf_models[name] = model
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items()})

t0 = time.time()
lr = LogisticRegression(featuresCol="features", labelCol="label", weightCol="weight", family="multinomial", maxIter=60, regParam=1e-3, elasticNetParam=0.0)
eval_clf("Logistic Regression", lr.fit(tr_c), time.time() - t0)

t0 = time.time()
rf = RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="weight", numTrees=RF_TREES, maxDepth=RF_DEPTH, seed=SEED)
eval_clf("Random Forest", rf.fit(tr_c), time.time() - t0)

if RUN_GBT_CLASSIFIER:
    t0 = time.time()
    gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=15, maxDepth=5, seed=SEED)
    ovr = OneVsRest(classifier=gbt, featuresCol="features", labelCol="label", weightCol="weight")
    eval_clf("GBT (One-vs-Rest)", ovr.fit(tr_c), time.time() - t0)

clf_df = pd.DataFrame(clf_results).set_index("model"); display(clf_df.round(4))
clf_df.to_csv(f"{OUT_DIR}/classification_results.csv")

In [ ]:
# Model comparison + confusion matrix + per-class F1 of the best model (by macro-F1)
best = clf_df["macro_f1"].idxmax(); print("Best classifier:", best)
fig, ax = plt.subplots(figsize=(8, 3.8))
clf_df[["accuracy","weighted_f1","macro_f1"]].plot.bar(ax=ax, color=["#2b6cb0","#2f855a","#c05621"], rot=0)
ax.set_ylim(0, 1); ax.set_title("Classification model comparison (test set)"); ax.set_xlabel(""); ax.legend(loc="upper right")
savefig("09_classification_comparison")

pdf = clf_preds[best].select("label", "prediction").toPandas()
labels_present = sorted(pdf.label.unique())
cm = confusion_matrix(pdf.label, pdf.prediction, labels=labels_present, normalize="true")
names = [CLASS_NAMES[int(i)] for i in labels_present]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax, annot_kws={"size": 7})
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"Normalized confusion matrix – {best}")
savefig("10_confusion_matrix")

f1s = f1_score(pdf.label, pdf.prediction, average=None, labels=labels_present)
fig, ax = plt.subplots(figsize=(7, 4.5)); order = np.argsort(f1s)
ax.barh(np.array(names)[order], f1s[order], color="#2b6cb0"); ax.set_xlabel("F1"); ax.set_title(f"Per-class F1 – {best}")
savefig("11_per_class_f1")

In [ ]:
# Feature importance (Random Forest classifier)
imp = clf_models["Random Forest"].featureImportances.toArray()
fi = pd.DataFrame({"feature": FEATURE_NAMES, "importance": imp}).sort_values("importance", ascending=False)
fi.head(50).to_csv(f"{OUT_DIR}/classification_feature_importance.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 6)); sns.barplot(data=fi.head(20), y="feature", x="importance", color="#2b6cb0", ax=ax)
ax.set_title("Top 20 features – category classification (Random Forest)"); ax.set_ylabel("")
savefig("12_classification_importance")

## 6. Research Question 2 – View-count regression (Modeling & Evaluation)
Target = `log(1 + views)` (views are heavily right-skewed). Models: **Linear Regression** (baseline, elastic-net), **Random Forest Regressor**, **GBT Regressor**. Metrics: RMSE, MAE, R² on the log scale; RMSE is also shown in original view units.

In [ ]:
reg_results, reg_models, reg_preds = [], {}, {}
ev = lambda m: RegressionEvaluator(labelCol="log_views", predictionCol="prediction", metricName=m)

def eval_reg(name, model, secs):
    pred = model.transform(test).cache(); reg_preds[name] = pred
    rmse_views = RegressionEvaluator(labelCol="views_d", predictionCol="pred_views", metricName="rmse").evaluate(
        pred.withColumn("views_d", F.col("views").cast("double")).withColumn("pred_views", F.expm1("prediction")))
    res = {"model": name, "RMSE_log": ev("rmse").evaluate(pred), "MAE_log": ev("mae").evaluate(pred),
           "R2_log": ev("r2").evaluate(pred), "RMSE_views": rmse_views, "train_sec": round(secs, 1)}
    reg_results.append(res); reg_models[name] = model
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items()})

t0 = time.time()
eval_reg("Linear Regression", LinearRegression(featuresCol="features", labelCol="log_views", maxIter=100, regParam=0.01, elasticNetParam=0.5).fit(train), time.time() - t0)
t0 = time.time()
eval_reg("Random Forest", RandomForestRegressor(featuresCol="features", labelCol="log_views", numTrees=RF_TREES, maxDepth=RF_DEPTH, seed=SEED).fit(train), time.time() - t0)
t0 = time.time()
eval_reg("GBT Regressor", GBTRegressor(featuresCol="features", labelCol="log_views", maxIter=50, maxDepth=6, seed=SEED).fit(train), time.time() - t0)

reg_df = pd.DataFrame(reg_results).set_index("model"); display(reg_df.round(4))
reg_df.to_csv(f"{OUT_DIR}/regression_results.csv")

In [ ]:
# Optional hyper-parameter tuning (3-fold CV on the Random Forest regressor). NOTE: CV folds are random rows,
# so repeated records of one video are already de-duplicated here (one row per video-country).
if RUN_TUNING:
    rf_r = RandomForestRegressor(featuresCol="features", labelCol="log_views", seed=SEED)
    grid = ParamGridBuilder().addGrid(rf_r.numTrees, [40, 80]).addGrid(rf_r.maxDepth, [8, 12]).build()
    cv = CrossValidator(estimator=rf_r, estimatorParamMaps=grid, evaluator=ev("rmse"), numFolds=3, parallelism=2, seed=SEED)
    t0 = time.time(); cvm = cv.fit(train)
    print("CV RMSE by config:", [round(x, 4) for x in cvm.avgMetrics]); print("Best params:", cvm.getEstimatorParamMaps()[int(np.argmin(cvm.avgMetrics))])
    eval_reg("Random Forest (tuned)", cvm.bestModel, time.time() - t0)
    reg_df = pd.DataFrame(reg_results).set_index("model"); display(reg_df.round(4))

In [ ]:
# Regression visuals: comparison, actual vs predicted, residuals, importance
best_r = reg_df["RMSE_log"].idxmin(); print("Best regressor:", best_r)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
reg_df["RMSE_log"].plot.bar(ax=ax[0], color="#c05621", rot=15); ax[0].set_title("RMSE (log scale) – lower is better"); ax[0].set_xlabel("")
reg_df["R2_log"].plot.bar(ax=ax[1], color="#2f855a", rot=15); ax[1].set_title("R² (log scale) – higher is better"); ax[1].set_xlabel("")
savefig("13_regression_comparison")

ps = reg_preds[best_r].select("log_views", "prediction").sample(min(1.0, 30000 / max(1, test.count())), seed=SEED).toPandas()
ps["resid"] = ps.log_views - ps.prediction
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].hexbin(ps.log_views, ps.prediction, gridsize=45, cmap="Blues", mincnt=1)
lim = [ps.log_views.min(), ps.log_views.max()]; ax[0].plot(lim, lim, "r--", lw=1); ax[0].set_xlabel("Actual log(1+views)"); ax[0].set_ylabel("Predicted"); ax[0].set_title(f"Actual vs predicted – {best_r}")
sns.histplot(ps.resid, bins=50, color="#2b6cb0", ax=ax[1]); ax[1].set_title("Residuals (actual − predicted)"); ax[1].set_xlabel("Residual (log scale)")
savefig("14_actual_vs_predicted")

rf_imp = reg_models["Random Forest"].featureImportances.toArray()
fr = pd.DataFrame({"feature": FEATURE_NAMES, "importance": rf_imp}).sort_values("importance", ascending=False)
fr.head(50).to_csv(f"{OUT_DIR}/regression_feature_importance.csv", index=False)
lr_coef = pd.DataFrame({"feature": FEATURE_NAMES, "coef": reg_models["Linear Regression"].coefficients.toArray()})
lr_top = lr_coef.reindex(lr_coef.coef.abs().sort_values(ascending=False).index).head(15)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
sns.barplot(data=fr.head(20), y="feature", x="importance", color="#2b6cb0", ax=ax[0]); ax[0].set_title("Top 20 features – view count (Random Forest)"); ax[0].set_ylabel("")
sns.barplot(data=lr_top, y="feature", x="coef", color="#c05621", ax=ax[1]); ax[1].set_title("Largest Linear Regression coefficients (standardized)"); ax[1].set_ylabel("")
savefig("15_regression_importance")

## 7. Save outputs
Writes the cleaned dataset (Parquet), result tables (CSV) and figures; zips figures and results so you can download them for the report.

In [ ]:
videos.drop("description", "thumbnail_link").write.mode("overwrite").parquet(f"{OUT_DIR}/videos_clean.parquet")
shutil.make_archive("/content/project_outputs", "zip", "/content", "figures")
shutil.make_archive("/content/project_results", "zip", OUT_DIR)
print("Saved: /content/project_outputs.zip (figures), /content/project_results.zip (tables + clean data)")
try:
    from google.colab import files
    files.download("/content/project_outputs.zip"); files.download("/content/project_results.zip")
except ImportError:
    pass